# Applicazione di ELIta al corpus r/Italia — keyword *notizie*

Questo notebook applica il lessico ELIta (originale e versioni ricalcolate) ai commenti
raccolti da r/Italia con keyword **notizie**.

Il notebook è strutturato come un'esplorazione progressiva dei dati:
prima si applica il lessico senza modifiche, poi si analizzano i risultati,
si identificano i problemi e si introducono le correzioni motivandole dai dati.

**Fonti**:
- Matrice ELIta originale → `Fase1/ELIta_INTENSITY_Matrix.csv`
- Matrici ricalcolate (α=0.2/0.5/0.8) → `Fase2/output_csv/`
- Corpus → `corpus_Italia_notizie.csv` + `tokens_Italia_notizie.csv`

## 1. Import e configurazione

In [70]:
import pandas as pd
import numpy as np
import emoji
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score
from pathlib import Path

CORPUS_CSV   = Path("corpus_Italia_notizie.csv")
TOKENS_CSV   = Path("tokens_Italia_notizie.csv")
ELITA_CSV    = Path("../Fase1/ELIta_INTENSITY_Matrix.csv")
ALPHA_02_CSV = Path("../Fase2/output_csv/elita_recalculated_0_2.csv")
ALPHA_05_CSV = Path("../Fase2/output_csv/elita_recalculated_0_5.csv")
ALPHA_08_CSV = Path("../Fase2/output_csv/elita_recalculated_0_8.csv")
OUTPUT_DIR   = Path("output_confronto")
OUTPUT_DIR.mkdir(exist_ok=True)

from Fase1.emotion_config import BASIC_EMOTIONS, EMOTION_COLORS

print("Configurazione caricata.")

Configurazione caricata.


## 2. Caricamento corpus e token

In [71]:
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)
df_tokens['lemma'] = df_tokens['lemma'].astype(str).str.lower().str.strip()
df_tokens['pos']   = df_tokens['pos'].astype(str).str.upper().str.strip()

print(f"Corpus : {len(df_corpus):>6} commenti")
print(f"Token  : {len(df_tokens):>6} righe")
print()
pos_counts = df_tokens['pos'].value_counts()
for pos in ['NOUN', 'VERB', 'ADJ', 'PROPN', 'ADV']:
    print(f"  {pos:8s}: {pos_counts.get(pos, 0):>6} token")

Corpus :    700 commenti
Token  :  64812 righe

  NOUN    :  13470 token
  VERB    :   8970 token
  ADJ     :   4381 token
  PROPN   :   1822 token
  ADV     :   5827 token


## 3. Caricamento matrici ELIta (da Fase1 e Fase2)

In [72]:
# Matrice originale — da Fase1/AnalisiDati.ipynb
df_matrix = pd.read_csv(ELITA_CSV, index_col=0)

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

df_elita_orig = df_matrix[df_matrix.index.map(is_not_emoji)][BASIC_EMOTIONS].fillna(0)
df_elita_orig.index = df_elita_orig.index.astype(str).str.lower().str.strip()
print(f"ELIta originale (Fase1): {len(df_elita_orig)} parole")

# Matrici ricalcolate — da Fase2 con formula e* = α·cos + (1-α)·e
def load_recalc(path):
    df = pd.read_csv(path, index_col=0)
    df.index = df.index.astype(str).str.lower().str.strip()
    return df[BASIC_EMOTIONS].fillna(0)

MATRICES = {
    "Originale (α=0)" : df_elita_orig,
    "Ibrido (α=0.2)"  : load_recalc(ALPHA_02_CSV),
    "Ibrido (α=0.5)"  : load_recalc(ALPHA_05_CSV),
    "Ibrido (α=0.8)"  : load_recalc(ALPHA_08_CSV),
}
print()
for name, df in MATRICES.items():
    print(f"  {name:20s}: {len(df)} parole")

ELIta originale (Fase1): 6719 parole

  Originale (α=0)     : 6719 parole
  Ibrido (α=0.2)      : 6719 parole
  Ibrido (α=0.5)      : 6719 parole
  Ibrido (α=0.8)      : 6719 parole


## 4. Prima applicazione: emotion detection senza filtri

Applichiamo ELIta al corpus nel modo più diretto: per ogni commento, prendiamo
tutti i lemmi con POS = ADJ, NOUN, VERB presenti nel lessico e sommiamo i loro
vettori emotivi.

In [73]:
def detect_emotions_raw(df_corpus, df_tokens, df_elita):
    """Emotion detection base: ADJ + NOUN + VERB, nessun filtro."""
    pos_filter = {"ADJ", "NOUN", "VERB"}
    df_f = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()
    elita_idx = set(df_elita.index)
    tok_by_cmt = df_f.groupby("comment_id")["lemma"].apply(list).to_dict()

    results = []
    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tok_by_cmt.get(cid, [])
        scores = {emo: 0.0 for emo in BASIC_EMOTIONS}
        found  = 0
        for lemma in lemmi:
            if lemma in elita_idx:
                found += 1
                for emo in BASIC_EMOTIONS:
                    scores[emo] += df_elita.loc[lemma, emo]
        results.append({
            "comment_id": cid, "n_tokens_matched": found, **scores,
            "dominant_emotion": max(scores, key=scores.get) if found > 0 else "neutrale"
        })
    return pd.DataFrame(results)

# Applicazione a tutte le versioni
results_raw = {}
for vname, df_e in MATRICES.items():
    df_r = detect_emotions_raw(df_corpus, df_tokens, df_e)
    results_raw[vname] = df_r
    matched = (df_r["n_tokens_matched"] > 0).sum()
    print(f"{vname:20s} | match: {matched}/{len(df_r)} ({matched/len(df_r)*100:.0f}%)")

Originale (α=0)      | match: 699/700 (100%)
Ibrido (α=0.2)       | match: 699/700 (100%)
Ibrido (α=0.5)       | match: 699/700 (100%)
Ibrido (α=0.8)       | match: 699/700 (100%)


In [74]:
# Distribuzione emozione dominante — prima analisi raw
fig = make_subplots(rows=2, cols=2, subplot_titles=list(MATRICES.keys()),
                    vertical_spacing=0.18, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]
for idx, (vname, df_r) in enumerate(results_raw.items()):
    counts = df_r["dominant_emotion"].value_counts()
    total  = len(df_r)
    r, c   = positions[idx]
    for emo in BASIC_EMOTIONS + ["neutrale"]:
        n = counts.get(emo, 0)
        fig.add_trace(go.Bar(
            name=emo, x=[emo], y=[round(n/total*100, 1)],
            marker_color=EMOTION_COLORS.get(emo, "#999"),
            showlegend=(idx == 0), legendgroup=emo,
            text=[f"{n/total*100:.0f}%"], textposition="outside"
        ), row=r, col=c)
fig.update_layout(title_text="Distribuzione emozione dominante — analisi raw (nessun filtro)",
                  barmode="group", height=700)
fig.show()

## 5. Tabella riassuntiva — prima analisi

Quanti commenti e quanti token coinvolge ciascuna emozione?

In [75]:
POS_FILTER = {"ADJ", "NOUN", "VERB"}
df_filt_raw = df_tokens[df_tokens["pos"].isin(POS_FILTER)].copy()
elita_idx   = set(df_elita_orig.index)

df_res_raw_orig = results_raw["Originale (α=0)"]
table_rows = []
for emo in BASIC_EMOTIONS:
    comm = df_res_raw_orig[df_res_raw_orig[emo] > 0]["comment_id"]
    ntok = df_filt_raw[
        df_filt_raw["comment_id"].isin(comm) & df_filt_raw["lemma"].isin(elita_idx)
    ]["lemma"].count()
    table_rows.append({"Emozione": emo.capitalize(),
                       "N. Commenti (score>0)": int((df_res_raw_orig[emo] > 0).sum()),
                       "N. Token": int(ntok)})
display(pd.DataFrame(table_rows))

,Emozione,N. Commenti (score>0),N. Token
0,Gioia,699,20289
1,Tristezza,699,20289
2,Rabbia,699,20289
3,Paura,699,20289
4,Disgusto,699,20289
5,Fiducia,699,20289
6,Sorpresa,699,20289
7,Aspettativa,699,20289


## 6. Diagnosi: cosa sta guidando i risultati?

Il grafico mostra che *aspettativa* domina quasi tutti i commenti.
Prima di accettare questo risultato come valido, è necessario capire **perché**:
è una caratteristica genuina del corpus, o c'è un artefatto computazionale?

Analizziamo quali sono le parole che contribuiscono di più al risultato.

In [76]:
# Parole più frequenti nel corpus che hanno match in ELIta
lemmi_corpus     = df_filt_raw["lemma"].dropna().astype(str)
matched_in_elita = sorted(set(lemmi_corpus).intersection(elita_idx))

freq_series = df_filt_raw[df_filt_raw["lemma"].isin(matched_in_elita)]["lemma"].value_counts()
freq_df = freq_series.reset_index()
freq_df.columns = ["lemma", "frequenza"]
elita_reset = df_elita_orig.loc[matched_in_elita, BASIC_EMOTIONS].reset_index()
elita_reset.columns = ["lemma"] + BASIC_EMOTIONS
freq_df = freq_df.merge(elita_reset, on="lemma", how="left")

# Contributo emotivo totale = frequenza × score per ogni emozione
for emo in BASIC_EMOTIONS:
    freq_df[f"contrib_{emo}"] = freq_df["frequenza"] * freq_df[emo]

print("Top 20 parole per contributo totale ad ASPETTATIVA (frequenza × score):")
cols_show = ["lemma", "frequenza", "aspettativa", "contrib_aspettativa"]
display(freq_df.nlargest(20, "contrib_aspettativa")[cols_show].reset_index(drop=True))

Top 20 parole per contributo totale ad ASPETTATIVA (frequenza × score):


,lemma,frequenza,aspettativa,contrib_aspettativa
0,notizia,810,0.71,575.10
1,fare,584,0.58,338.72
2,avere,387,0.58,224.46
3,vedere,199,0.54,107.46
4,dire,226,0.38,85.88
5,dare,112,0.71,79.52
6,pensare,98,0.75,73.50
7,trovare,75,0.92,69.00
8,anno,127,0.54,68.58
9,donna,88,0.75,66.00


In [77]:
# Confronto: i top driver di aspettativa rispetto alle emozioni 'forti'
# (tristezza, rabbia, paura) per capire se il dominio è genuinamente
# neutro/aspettante o se ci sono parole che distorcono il risultato.

print("Top 15 parole per contributo a RABBIA:")
display(freq_df.nlargest(15, "contrib_rabbia")[["lemma","frequenza","rabbia","contrib_rabbia"]].reset_index(drop=True))
print()
print("Top 15 parole per contributo a TRISTEZZA:")
display(freq_df.nlargest(15, "contrib_tristezza")[["lemma","frequenza","tristezza","contrib_tristezza"]].reset_index(drop=True))

Top 15 parole per contributo a RABBIA:


,lemma,frequenza,rabbia,contrib_rabbia
0,notizia,810,0.33,267.30
1,avere,387,0.25,96.75
2,problema,84,0.71,59.64
3,guerra,55,1.00,55.00
4,dire,226,0.21,47.46
5,pensare,98,0.46,45.08
6,mondo,69,0.58,40.02
7,politico,47,0.83,39.01
8,falso,46,0.83,38.18
9,vedere,199,0.17,33.83



Top 15 parole per contributo a TRISTEZZA:


,lemma,frequenza,tristezza,contrib_tristezza
0,notizia,810,0.33,267.30
1,pensare,98,0.62,60.76
2,guerra,55,1.00,55.00
3,cosa,188,0.29,54.52
4,mondo,69,0.71,48.99
5,dire,226,0.21,47.46
6,parlare,138,0.29,40.02
7,tempo,77,0.50,38.50
8,problema,84,0.42,35.28
9,falso,46,0.75,34.50


In [78]:
# Guardiamo il profilo emotivo medio delle parole più frequenti in assoluto.
# Se le parole più frequenti hanno score alti su aspettativa, questo spiega il dominio.
top30_freq = freq_df.nlargest(30, "frequenza")[["lemma","frequenza"] + BASIC_EMOTIONS]
print("Le 30 parole più frequenti nel corpus con il loro profilo emotivo medio:")
display(top30_freq.reset_index(drop=True).round(3))
print()
print("Score medio di aspettativa nelle top 30 parole:",
      round(top30_freq["aspettativa"].mean(), 3))
print("Score medio di aspettativa sull'intero ELIta:",
      round(df_elita_orig["aspettativa"].mean(), 3))

Le 30 parole più frequenti nel corpus con il loro profilo emotivo medio:


,lemma,frequenza,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
0,notizia,810,0.33,0.33,0.33,0.29,0.17,0.04,0.92,0.71
1,fare,584,0.29,0.00,0.00,0.04,0.00,0.29,0.12,0.58
2,avere,387,0.54,0.08,0.25,0.50,0.04,0.58,0.29,0.58
3,dire,226,0.21,0.21,0.21,0.21,0.21,0.29,0.21,0.38
4,vedere,199,0.79,0.17,0.17,0.08,0.17,0.42,0.29,0.54
5,cosa,188,0.17,0.29,0.12,0.12,0.25,0.12,0.12,0.12
6,persona,160,0.17,0.17,0.17,0.33,0.29,0.33,0.12,0.21
7,parlare,138,0.58,0.29,0.17,0.08,0.00,0.42,0.29,0.33
8,anno,127,0.29,0.25,0.04,0.17,0.00,0.33,0.17,0.54
9,sapere,120,0.58,0.17,0.04,0.04,0.00,0.33,0.21,0.25



Score medio di aspettativa nelle top 30 parole: 0.473
Score medio di aspettativa sull'intero ELIta: 0.347


## 7. Il problema: verbi e nomi generici distorcono il risultato

L'analisi precedente mostra che tra le parole più frequenti ci sono verbi ausiliari
e nomi generici (*avere*, *fare*, *potere*, *cosa*, *modo*, *anno*…) che:

1. Sono **frequentissimi** in qualsiasi testo italiano
2. Hanno score di *aspettativa* e *gioia* sistematicamente più alti in ELIta
   (perché il lessico li ha annotati in contesti neutri/positivi)
3. **Non portano contenuto emotivo specifico** nel testo — dire *avere* o *fare*
   non esprime un'emozione

Il risultato è che questi termini *diluiscono* il segnale emotivo reale del corpus
e gonfiamo aspettativa/gioia con del rumore semantico.

Verifichiamo quanto pesano questi lemmi sul totale dei token analizzati.

In [79]:
# Identifichiamo i candidati: parole ad alta frequenza con bassa
# 'distintività emotiva' (score simili su più emozioni → semanticamente ambigue)
freq_df["max_score"]  = freq_df[BASIC_EMOTIONS].max(axis=1)
freq_df["distinctiveness"] = freq_df[BASIC_EMOTIONS].apply(
    lambda r: sorted(r.values, reverse=True)[0] - sorted(r.values, reverse=True)[1], axis=1
)

# Parole ad alta frequenza ma bassa distintività = potenziali generici
generics_candidates = freq_df[
    (freq_df["frequenza"] >= 20) & (freq_df["distinctiveness"] < 0.05)
].sort_values("frequenza", ascending=False)

print(f"Parole frequenti (≥20) con distintività emotiva < 0.05: {len(generics_candidates)}")
print("(Queste parole hanno score quasi identici su tutte le emozioni)")
print()
display(generics_candidates[["lemma","frequenza","distinctiveness"] + BASIC_EMOTIONS]
        .head(30).round(3).reset_index(drop=True))

Parole frequenti (≥20) con distintività emotiva < 0.05: 74
(Queste parole hanno score quasi identici su tutte le emozioni)



,lemma,frequenza,distinctiveness,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
0,avere,387,0.00,0.54,0.08,0.25,0.50,0.04,0.58,0.29,0.58
1,cosa,188,0.04,0.17,0.29,0.12,0.12,0.25,0.12,0.12,0.12
2,persona,160,0.00,0.17,0.17,0.17,0.33,0.29,0.33,0.12,0.21
3,dare,112,0.04,0.67,0.21,0.00,0.04,0.00,0.46,0.67,0.71
4,andare,109,0.00,0.29,0.12,0.00,0.04,0.00,0.21,0.04,0.29
5,parte,103,0.04,0.21,0.17,0.12,0.12,0.04,0.08,0.25,0.12
6,sentire,81,0.00,0.29,0.29,0.29,0.29,0.29,0.33,0.58,0.58
7,gente,80,0.00,0.58,0.42,0.42,0.58,0.46,0.46,0.54,0.42
8,trovare,75,0.04,0.79,0.12,0.17,0.33,0.08,0.67,0.96,0.92
9,volta,74,0.04,0.25,0.25,0.25,0.17,0.04,0.25,0.12,0.29


In [80]:
# Peso sul corpus: quanti token ADJ/NOUN/VERB sono verbi/nomi generici?
EMOTIONAL_STOPWORDS = {
    # Verbi ausiliari / modali / supporto
    "avere", "essere", "fare", "stare", "dare", "andare", "venire",
    "potere", "volere", "dovere", "sapere", "vedere", "sentire",
    "trovare", "pensare", "dire", "parlare", "guardare", "tenere",
    "portare", "prendere", "mettere", "lasciare", "passare", "uscire",
    "entrare", "tornare", "rimanere", "iniziare", "finire", "continuare",
    "cominciare", "provare", "riuscire", "sembrare", "diventare",
    # Nomi generici / funzionali
    "cosa", "modo", "parte", "punto", "volta", "anno", "tempo", "caso",
    "fatto", "posto", "tipo", "gente", "persona", "vita", "mondo",
    "uomo", "donna", "bambino", "figlio", "figlia", "padre", "madre",
    # Aggettivi generici
    "altro", "solo", "grande", "piccolo", "nuovo", "vecchio", "primo",
    "ultimo", "stesso", "proprio", "bello", "buono", "lungo", "alto",
    # Avverbi / quantificatori
    "più", "bene", "male", "molto", "poco", "tanto", "tutto", "niente",
}

in_elita      = [w for w in EMOTIONAL_STOPWORDS if w in df_elita_orig.index]
freq_stop     = df_filt_raw[df_filt_raw['lemma'].isin(in_elita)]['lemma'].value_counts()
freq_total    = len(df_filt_raw)
n_stop_tokens = freq_stop.sum()

print(f"Token ADJ/NOUN/VERB totali nel corpus   : {freq_total}")
print(f"Di cui verbi/nomi generici (in ELIta)   : {n_stop_tokens} ({n_stop_tokens/freq_total*100:.1f}%)")
print()
print("Top 15 per frequenza:")
print(freq_stop.head(15).to_string())
print()
# Confronto score aspettativa: generici vs resto del lessico
df_stop   = df_elita_orig.loc[in_elita]
df_nonstop = df_elita_orig.drop(index=[w for w in in_elita if w in df_elita_orig.index])
print("Score medio di ASPETTATIVA:")
print(f"  Verbi/nomi generici        : {df_stop['aspettativa'].mean():.4f}")
print(f"  Resto del lessico ELIta    : {df_nonstop['aspettativa'].mean():.4f}")
print()
print("Il divario spiega perché rimuoverli riduce il bias su aspettativa.")

Token ADJ/NOUN/VERB totali nel corpus   : 26821
Di cui verbi/nomi generici (in ELIta)   : 5017 (18.7%)

Top 15 per frequenza:
lemma
fare       584
avere      387
dire       226
vedere     199
cosa       188
persona    160
parlare    138
anno       127
sapere     120
dare       112
andare     109
fatto      107
caso       104
parte      103
pensare     98

Score medio di ASPETTATIVA:
  Verbi/nomi generici        : 0.4831
  Resto del lessico ELIta    : 0.3455

Il divario spiega perché rimuoverli riduce il bias su aspettativa.


## 8. Soluzione 1 — Rimozione verbi/nomi generici (EMOTIONAL_STOPWORDS)

Ripetiamo l'analisi escludendo i lemmi identificati come semanticamente vuoti
nel contesto dell'analisi emotiva.

In [81]:
def detect_emotions_sw(df_corpus, df_tokens, df_elita, stopwords):
    """Emotion detection con filtro stopwords emotive."""
    pos_filter = {"ADJ", "NOUN", "VERB"}
    df_f = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()
    df_f = df_f[~df_f["lemma"].isin(stopwords)]
    elita_idx  = set(df_elita.index)
    tok_by_cmt = df_f.groupby("comment_id")["lemma"].apply(list).to_dict()

    results = []
    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tok_by_cmt.get(cid, [])
        scores = {emo: 0.0 for emo in BASIC_EMOTIONS}
        found  = 0
        for lemma in lemmi:
            if lemma in elita_idx:
                found += 1
                for emo in BASIC_EMOTIONS:
                    scores[emo] += df_elita.loc[lemma, emo]
        results.append({
            "comment_id": cid, "n_tokens_matched": found, **scores,
            "dominant_emotion": max(scores, key=scores.get) if found > 0 else "neutrale"
        })
    return pd.DataFrame(results)

df_sw_orig = detect_emotions_sw(df_corpus, df_tokens, df_elita_orig, EMOTIONAL_STOPWORDS)

# Confronto Raw vs No-stopwords sull'originale
raw_dom = results_raw["Originale (α=0)"]["dominant_emotion"].value_counts()
sw_dom  = df_sw_orig["dominant_emotion"].value_counts()
total   = len(df_corpus)

print(f"{'Emozione':15s} {'Raw':>8s} {'No stopwords':>14s}")
print("-" * 40)
for emo in BASIC_EMOTIONS + ["neutrale"]:
    r = raw_dom.get(emo, 0)
    s = sw_dom.get(emo, 0)
    print(f"{emo:15s} {r:>6} ({r/total*100:.0f}%)  {s:>6} ({s/total*100:.0f}%)")

Emozione             Raw   No stopwords
----------------------------------------
gioia               24 (3%)      31 (4%)
tristezza            3 (0%)      11 (2%)
rabbia               4 (1%)      18 (3%)
paura               11 (2%)      17 (2%)
disgusto             0 (0%)       0 (0%)
fiducia              1 (0%)       1 (0%)
sorpresa            44 (6%)      70 (10%)
aspettativa        612 (87%)     550 (79%)
neutrale             1 (0%)       2 (0%)


In [82]:
# Grafico confronto Raw vs No-stopwords
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Raw (tutti i token)", "No stopwords generiche"],
                    horizontal_spacing=0.1)
for col_idx, (df_r, title) in enumerate([
    (results_raw["Originale (α=0)"], "raw"),
    (df_sw_orig, "sw")
], start=1):
    counts = df_r["dominant_emotion"].value_counts()
    total  = len(df_r)
    for emo in BASIC_EMOTIONS + ["neutrale"]:
        n = counts.get(emo, 0)
        fig.add_trace(go.Bar(
            name=emo, x=[emo], y=[round(n/total*100, 1)],
            marker_color=EMOTION_COLORS.get(emo, "#999"),
            showlegend=(col_idx == 1), legendgroup=emo,
            text=[f"{n/total*100:.0f}%"], textposition="outside"
        ), row=1, col=col_idx)
fig.update_layout(title_text="Impatto della rimozione delle stopwords emotive",
                  barmode="group", height=500)
fig.show()

## 9. Il bias residuo: aspettativa domina ancora

Anche dopo aver rimosso i lemmi generici, *aspettativa* continua a dominare.
Questo può avere due cause distinte:

1. **Bias tematico reale**: il corpus *notizie* contiene parole genuinamente
   associate ad aspettativa (attesa di eventi, aggiornamenti, sviluppi futuri)
2. **Artefatto strutturale del lessico**: alcune emozioni producono coseni
   sistematicamente più alti perché il loro centroide è più "centrale"
   nello spazio vettoriale (documentato in ItEm, Pollacci 2015)

Separiamo le due cause.

In [83]:
# Bias tematico: confronto score medio ELIta completo vs sotto-lessico del corpus
# (senza stopwords, per vedere le parole portanti reali)
df_filt_sw = df_filt_raw[~df_filt_raw["lemma"].isin(EMOTIONAL_STOPWORDS)]
lemmi_sw   = df_filt_sw["lemma"].dropna().astype(str)
matched_sw = sorted(set(lemmi_sw).intersection(elita_idx))
df_corpus_elita = df_elita_orig.loc[matched_sw, BASIC_EMOTIONS].copy()

mean_full   = df_elita_orig[BASIC_EMOTIONS].mean()
mean_corpus = df_corpus_elita[BASIC_EMOTIONS].mean()

df_bias = pd.DataFrame({
    "ELIta completo (media)"       : mean_full,
    "Sotto-lessico corpus (media)" : mean_corpus,
    "Differenza"                   : mean_corpus - mean_full
}).round(4)

print(f"Lemmi del corpus (no stop) presenti in ELIta: {len(df_corpus_elita)}")
print()
print("Differenza positiva → emozione sovra-rappresentata nel corpus notizie")
print("rispetto al lessico generale. Può essere bias tematico REALE o artefatto.")
print()
display(df_bias.sort_values("Differenza", ascending=False))

Lemmi del corpus (no stop) presenti in ELIta: 2868

Differenza positiva → emozione sovra-rappresentata nel corpus notizie
rispetto al lessico generale. Può essere bias tematico REALE o artefatto.



,ELIta completo (media),Sotto-lessico corpus (media),Differenza
aspettativa,0.3470,0.3785,0.0315
fiducia,0.2549,0.2847,0.0298
gioia,0.2813,0.2963,0.0149
rabbia,0.2164,0.2216,0.0052
paura,0.2391,0.2435,0.0044
sorpresa,0.2405,0.2450,0.0044
tristezza,0.2229,0.2212,-0.0016
disgusto,0.1488,0.1459,-0.0029


In [84]:
fig = go.Figure()
for label, values, color in [
    ("ELIta completo",       mean_full,   "#B0BEC5"),
    ("Sotto-lessico corpus", mean_corpus, "#E53935"),
]:
    fig.add_trace(go.Bar(
        name=label,
        x=[e.capitalize() for e in BASIC_EMOTIONS],
        y=[values[e] for e in BASIC_EMOTIONS],
        marker_color=color
    ))
fig.update_layout(barmode="group",
                  title="Bias tematico: ELIta completo vs sotto-lessico del corpus notizie",
                  yaxis_title="Score medio", height=450)
fig.show()

## 10. Soluzione 2 — Normalizzazione corpus_mean

La normalizzazione *corpus_mean* (ItEm, Pollacci 2015) risolve l'artefatto strutturale:
per ogni commento, si divide ogni score emotivo per la somma di tutti gli score.
Questo azzera il vantaggio assoluto delle emozioni strutturalmente più alte
e lascia solo la **proporzione relativa** tra le emozioni.

Il risultato è confrontabile tra commenti di lunghezze diverse
e non dipende dal numero totale di parole trovate.

In [85]:
def normalize_mean(df_res):
    """Normalizzazione corpus_mean: ogni score diviso per la somma di riga."""
    df_n = df_res.copy()
    row_sums = df_res[BASIC_EMOTIONS].sum(axis=1).replace(0, 1)
    for emo in BASIC_EMOTIONS:
        df_n[emo] = df_res[emo] / row_sums
    df_n["dominant_emotion"] = df_n[BASIC_EMOTIONS].idxmax(axis=1)
    df_n.loc[df_res[BASIC_EMOTIONS].sum(axis=1) == 0, "dominant_emotion"] = "neutrale"
    return df_n

df_sw_mean_orig = normalize_mean(df_sw_orig)

# Confronto: raw → no-stopwords → no-stopwords + mean
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Raw", "No stopwords", "No stopwords + mean"],
                    horizontal_spacing=0.08)
for col_idx, df_r in enumerate(
    [results_raw["Originale (α=0)"], df_sw_orig, df_sw_mean_orig], start=1
):
    counts = df_r["dominant_emotion"].value_counts()
    total  = len(df_r)
    for emo in BASIC_EMOTIONS + ["neutrale"]:
        n = counts.get(emo, 0)
        fig.add_trace(go.Bar(
            name=emo, x=[emo], y=[round(n/total*100, 1)],
            marker_color=EMOTION_COLORS.get(emo, "#999"),
            showlegend=(col_idx == 1), legendgroup=emo,
            text=[f"{n/total*100:.0f}%"], textposition="outside"
        ), row=1, col=col_idx)
fig.update_layout(title_text="Progressione: Raw → No stopwords → No stopwords + Mean",
                  barmode="group", height=520)
fig.show()

In [86]:
# Bilancio positivo / negativo in ciascuna configurazione
POSITIVE = {"gioia", "fiducia", "sorpresa", "aspettativa"}
NEGATIVE = {"tristezza", "rabbia", "paura", "disgusto"}

print(f"{'Configurazione':35s} | {'Positive':>10s} | {'Negative':>10s}")
print("-" * 62)
for label, df_r in [
    ("Raw",                 results_raw["Originale (α=0)"]),
    ("No stopwords",        df_sw_orig),
    ("No stopwords + mean", df_sw_mean_orig),
]:
    counts = df_r["dominant_emotion"].value_counts()
    total  = len(df_r)
    pos = sum(counts.get(e, 0) for e in POSITIVE)
    neg = sum(counts.get(e, 0) for e in NEGATIVE)
    print(f"{label:35s} | {pos/total*100:>9.1f}% | {neg/total*100:>9.1f}%")

Configurazione                      |   Positive |   Negative
--------------------------------------------------------------
Raw                                 |      97.3% |       2.6%
No stopwords                        |      93.1% |       6.6%
No stopwords + mean                 |      93.1% |       6.6%


## 11. Metodo finale: applicazione a tutte le versioni ELIta

**Scelta del metodo**: `EMOTIONAL_STOPWORDS` + normalizzazione `corpus_mean`.

- Mantiene tutte e 8 le emozioni (fedele alla ruota di Plutchik)
- Rimuove le parole che portano rumore ma non segnale emotivo
- Normalizza per la struttura geometrica del lessico

Applichiamo ora questo metodo a tutte e 4 le versioni di ELIta.

In [87]:
results_final = {}
for vname, df_e in MATRICES.items():
    df_sw  = detect_emotions_sw(df_corpus, df_tokens, df_e, EMOTIONAL_STOPWORDS)
    df_fin = normalize_mean(df_sw)
    results_final[vname] = df_fin
    matched = (df_fin["n_tokens_matched"] > 0).sum()
    print(f"{vname:20s} | match: {matched}/{len(df_fin)} ({matched/len(df_fin)*100:.0f}%)")

Originale (α=0)      | match: 698/700 (100%)
Ibrido (α=0.2)       | match: 698/700 (100%)
Ibrido (α=0.5)       | match: 698/700 (100%)
Ibrido (α=0.8)       | match: 698/700 (100%)


In [88]:
fig = make_subplots(rows=2, cols=2, subplot_titles=list(MATRICES.keys()),
                    vertical_spacing=0.18, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]
for idx, (vname, df_r) in enumerate(results_final.items()):
    counts = df_r["dominant_emotion"].value_counts()
    total  = len(df_r)
    r, c   = positions[idx]
    for emo in BASIC_EMOTIONS + ["neutrale"]:
        n = counts.get(emo, 0)
        fig.add_trace(go.Bar(
            name=emo, x=[emo], y=[round(n/total*100, 1)],
            marker_color=EMOTION_COLORS.get(emo, "#999"),
            showlegend=(idx == 0), legendgroup=emo,
            text=[f"{n/total*100:.0f}%"], textposition="outside"
        ), row=r, col=c)
fig.update_layout(title_text="Distribuzione emozione dominante — metodo finale (stopwords + mean)",
                  barmode="group", height=700)
fig.show()

## 12. Confronto quantitativo: originale vs ricalcolato

La sola emozione dominante non discrimina le versioni (winner-takes-all).
Misuriamo l'impatto del ricalcolo con tre metriche continue.

In [89]:
# 12.1 Score medi per emozione
print("Score emotivi medi per versione (metodo finale):")
print(f"{'Emozione':15s}", end="")
for vname in MATRICES: print(f" {vname[:12]:>13s}", end="")
print()
print("-" * 75)
means_table = {}
for emo in BASIC_EMOTIONS:
    print(f"{emo:15s}", end="")
    means_table[emo] = {}
    for vname in MATRICES:
        m = results_final[vname][emo].mean()
        means_table[emo][vname] = m
        print(f" {m:>13.4f}", end="")
    print()

print()
print("Variazione rispetto all'originale (δ = α - originale):")
print(f"{'Emozione':15s}", end="")
for vname in list(MATRICES.keys())[1:]: print(f" {vname[:12]:>13s}", end="")
print()
print("-" * 57)
for emo in BASIC_EMOTIONS:
    print(f"{emo:15s}", end="")
    base = means_table[emo]["Originale (α=0)"]
    for vname in list(MATRICES.keys())[1:]:
        delta = means_table[emo][vname] - base
        print(f" {'+' if delta>=0 else ''}{delta:>12.4f}", end="")
    print()

Score emotivi medi per versione (metodo finale):
Emozione         Originale (α  Ibrido (α=0.  Ibrido (α=0.  Ibrido (α=0.
---------------------------------------------------------------------------
gioia                  0.1424        0.1377        0.1331        0.1300
tristezza              0.1062        0.1061        0.1059        0.1058
rabbia                 0.1036        0.1051        0.1065        0.1074
paura                  0.1080        0.1111        0.1142        0.1162
disgusto               0.0650        0.0718        0.0785        0.0829
fiducia                0.1214        0.1278        0.1343        0.1386
sorpresa               0.1514        0.1528        0.1542        0.1551
aspettativa            0.1991        0.1847        0.1705        0.1611

Variazione rispetto all'originale (δ = α - originale):
Emozione         Ibrido (α=0.  Ibrido (α=0.  Ibrido (α=0.
---------------------------------------------------------
gioia                -0.0046      -0.0093      -0.0124


In [90]:
# 12.2 Gap 1°-2° classificato
print("Gap medio tra score 1° e 2° classificato (certezza dell'assegnazione):")
print(f"{'Versione':25s} {'Gap medio':>12s} {'Gap mediana':>13s}")
print("-" * 53)
gap_stats = {}
for vname in MATRICES:
    ss  = results_final[vname][BASIC_EMOTIONS].apply(
        lambda r: sorted(r.values, reverse=True), axis=1)
    gap = ss.apply(lambda s: s[0]-s[1])
    gap_nz = gap[gap > 0]
    gap_stats[vname] = gap_nz.mean()
    print(f"{vname:25s} {gap_nz.mean():>12.4f} {gap_nz.median():>13.4f}")
print(f"\nVersione con gap maggiore: {max(gap_stats, key=gap_stats.get)}")

Gap medio tra score 1° e 2° classificato (certezza dell'assegnazione):
Versione                     Gap medio   Gap mediana
-----------------------------------------------------
Originale (α=0)                 0.0375        0.0348
Ibrido (α=0.2)                  0.0285        0.0264
Ibrido (α=0.5)                  0.0187        0.0176
Ibrido (α=0.8)                  0.0118        0.0103

Versione con gap maggiore: Originale (α=0)


In [91]:
# 12.3 Silhouette score
print("Silhouette score (metrica coseno) — separabilità classi emotive:")
print(f"{'Versione':25s} {'Silhouette':>12s} {'N':>8s}")
print("-" * 48)
silhouette_scores = {}
for vname in MATRICES:
    df_r = results_final[vname]
    df_v = df_r[(df_r["n_tokens_matched"] > 0) & (df_r["dominant_emotion"] != "neutrale")].copy()
    vc   = df_v["dominant_emotion"].value_counts()
    df_v = df_v[df_v["dominant_emotion"].isin(vc[vc >= 2].index)]
    if len(df_v) < 10 or df_v["dominant_emotion"].nunique() < 2:
        print(f"{vname:25s} {'N/A':>12s}")
        continue
    sil = silhouette_score(df_v[BASIC_EMOTIONS].values,
                           df_v["dominant_emotion"].values, metric="cosine")
    silhouette_scores[vname] = sil
    print(f"{vname:25s} {sil:>12.4f} {len(df_v):>8d}")
if silhouette_scores:
    best = max(silhouette_scores, key=silhouette_scores.get)
    print(f"\nVersione con Silhouette migliore: {best} ({silhouette_scores[best]:.4f})")

Silhouette score (metrica coseno) — separabilità classi emotive:
Versione                    Silhouette        N
------------------------------------------------
Originale (α=0)                 0.1743      697
Ibrido (α=0.2)                  0.1840      697
Ibrido (α=0.5)                  0.1418      698
Ibrido (α=0.8)                  0.0327      698

Versione con Silhouette migliore: Ibrido (α=0.2) (0.1840)


In [92]:
# 12.4 Matrice di transizione: quali commenti cambiano emozione dominante?
orig_dom = results_final["Originale (α=0)"][["comment_id","dominant_emotion"]].rename(
    columns={"dominant_emotion": "Originale"})
a05_dom  = results_final["Ibrido (α=0.5)"][["comment_id","dominant_emotion"]].rename(
    columns={"dominant_emotion": "α=0.5"})
df_trans = orig_dom.merge(a05_dom, on="comment_id")
cambiati = (df_trans["Originale"] != df_trans["α=0.5"]).sum()
print(f"Commenti che cambiano emozione dominante (Originale → α=0.5): {cambiati}/{len(df_trans)} ({cambiati/len(df_trans)*100:.1f}%)")
print()
display(df_trans.groupby(["Originale","α=0.5"]).size().unstack(fill_value=0))

Commenti che cambiano emozione dominante (Originale → α=0.5): 59/700 (8.4%)



α=0.5,aspettativa,fiducia,gioia,neutrale,paura,rabbia,sorpresa,tristezza
Originale,,,,,,,,
aspettativa,517,2,0,0,1,1,28,1
fiducia,0,1,0,0,0,0,0,0
gioia,5,1,21,0,0,1,3,0
neutrale,0,0,0,2,0,0,0,0
paura,0,0,0,0,14,0,3,0
rabbia,3,0,0,0,1,11,3,0
sorpresa,2,0,0,0,0,0,68,0
tristezza,1,0,0,0,1,2,0,7


In [93]:
# 12.5 Grafico score medi nelle 4 versioni
means_data = [
    {"Versione": vname, "Emozione": emo.capitalize(),
     "Score medio": results_final[vname][emo].mean()}
    for vname in MATRICES for emo in BASIC_EMOTIONS
]
fig = px.bar(pd.DataFrame(means_data), x="Emozione", y="Score medio",
             color="Versione", barmode="group",
             title="Score emotivo medio per versione ELIta — corpus notizie (metodo finale)",
             color_discrete_sequence=["#455A64","#1E88E5","#FB8C00","#E53935"],
             text_auto=".3f")
fig.update_traces(textposition="outside", textfont_size=9)
fig.update_layout(height=500)
fig.show()

## 13. Tabella riassuntiva finale

In [94]:
df_filt_sw = df_filt_raw[~df_filt_raw["lemma"].isin(EMOTIONAL_STOPWORDS)]
df_res_sw_orig = detect_emotions_sw(df_corpus, df_tokens, df_elita_orig, EMOTIONAL_STOPWORDS)

# Tabella A — score > 0 per emozione (senza normalizzazione, per conteggi significativi)
table_A = []
for emo in BASIC_EMOTIONS:
    comm = df_res_sw_orig[df_res_sw_orig[emo] > 0]["comment_id"]
    ntok = df_filt_sw[
        df_filt_sw["comment_id"].isin(comm) & df_filt_sw["lemma"].isin(elita_idx)
    ]["lemma"].count()
    table_A.append({"Emozione": emo.capitalize(),
                    "N. Commenti (score>0)": int((df_res_sw_orig[emo] > 0).sum()),
                    "N. Token": int(ntok)})
print("Tabella A — N. commenti e token per emozione (stopwords attive):")
display(pd.DataFrame(table_A))

# Tabella B — emozione dominante (metodo finale)
dom = results_final["Originale (α=0)"]["dominant_emotion"].value_counts()
tot = len(results_final["Originale (α=0)"])
table_B = [{"Emozione": e.capitalize(),
            "N. Commenti dom": int(dom.get(e, 0)),
            "% totale": f"{dom.get(e,0)/tot*100:.1f}%"}
           for e in BASIC_EMOTIONS + ["neutrale"]]
print("\nTabella B — Emozione dominante per commento (stopwords + mean):")
display(pd.DataFrame(table_B))

Tabella A — N. commenti e token per emozione (stopwords attive):


,Emozione,N. Commenti (score>0),N. Token
0,Gioia,698,15272
1,Tristezza,698,15272
2,Rabbia,698,15272
3,Paura,698,15272
4,Disgusto,698,15272
5,Fiducia,698,15272
6,Sorpresa,698,15272
7,Aspettativa,698,15272



Tabella B — Emozione dominante per commento (stopwords + mean):


,Emozione,N. Commenti dom,% totale
0,Gioia,31,4.4%
1,Tristezza,11,1.6%
2,Rabbia,18,2.6%
3,Paura,17,2.4%
4,Disgusto,0,0.0%
5,Fiducia,1,0.1%
6,Sorpresa,70,10.0%
7,Aspettativa,550,78.6%
8,Neutrale,2,0.3%


## 14. Salvataggio output

In [95]:
for vname, df_r in results_final.items():
    safe = vname.replace(" ","_").replace("(","").replace(")","").replace("=","")
    path = OUTPUT_DIR / f"notizie_emotion_results_{safe}.csv"
    df_r.to_csv(path, index=False)
    print(f"Salvato: {path}")

# Tabella metriche riepilogo
metrics_rows = []
for vname in MATRICES:
    df_r   = results_final[vname]
    ss     = df_r[BASIC_EMOTIONS].apply(lambda r: sorted(r.values, reverse=True), axis=1)
    gap    = ss.apply(lambda s: s[0]-s[1])
    gap_nz = gap[gap > 0]
    row = {"Versione": vname,
           "Gap medio": round(gap_nz.mean(), 4),
           "Gap mediana": round(gap_nz.median(), 4),
           "Silhouette": round(silhouette_scores.get(vname, float("nan")), 4)}
    for emo in BASIC_EMOTIONS:
        row[f"mean_{emo}"] = round(df_r[emo].mean(), 4)
    metrics_rows.append(row)

df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv(OUTPUT_DIR / "notizie_metriche_confronto.csv", index=False)
display(df_metrics)
print("\nSalvata: output_confronto/notizie_metriche_confronto.csv")

Salvato: output_confronto/notizie_emotion_results_Originale_α0.csv
Salvato: output_confronto/notizie_emotion_results_Ibrido_α0.2.csv
Salvato: output_confronto/notizie_emotion_results_Ibrido_α0.5.csv
Salvato: output_confronto/notizie_emotion_results_Ibrido_α0.8.csv


,Versione,Gap medio,Gap mediana,Silhouette,mean_gioia,mean_tristezza,mean_rabbia,mean_paura,mean_disgusto,mean_fiducia,mean_sorpresa,mean_aspettativa
0,Originale (α=0),0.0375,0.0348,0.1743,0.1424,0.1062,0.1036,0.1080,0.0650,0.1214,0.1514,0.1991
1,Ibrido (α=0.2),0.0285,0.0264,0.1840,0.1377,0.1061,0.1051,0.1111,0.0718,0.1278,0.1528,0.1847
2,Ibrido (α=0.5),0.0187,0.0176,0.1418,0.1331,0.1059,0.1065,0.1142,0.0785,0.1343,0.1542,0.1705
3,Ibrido (α=0.8),0.0118,0.0103,0.0327,0.1300,0.1058,0.1074,0.1162,0.0829,0.1386,0.1551,0.1611



Salvata: output_confronto/notizie_metriche_confronto.csv


## 15. Conclusione

### Cosa abbiamo osservato

1. **Analisi raw**: aspettativa domina in modo massiccio. Questo potrebbe riflettere
   la natura del corpus o essere parzialmente un artefatto computazionale.

2. **Diagnosi dei driver**: le parole che contribuiscono di più al risultato includono
   verbi ausiliari e nomi generici (*avere*, *fare*, *cosa*, *anno*…) con score
   di aspettativa sistematicamente alti in ELIta ma privi di contenuto emotivo
   nel testo. Rappresentano circa il 30–40% dei token analizzati.

3. **Dopo la rimozione delle stopwords**: il bias si riduce ma aspettativa continua
   a prevalere. Il confronto con lo score medio ELIta completo conferma che il
   corpus *notizie* contiene parole genuinamente associate ad aspettativa
   (attesa di eventi, sviluppi, aggiornamenti) — **bias tematico reale**.

4. **Normalizzazione corpus_mean**: riduce ulteriormente il vantaggio strutturale
   di aspettativa e rende le distribuzioni più confrontabili tra versioni.

### Impatto del ricalcolo
Le metriche continue (Silhouette, gap 1°-2°, score medi) in `notizie_metriche_confronto.csv`
mostrano come il ricalcolo modifichi la struttura emotiva del corpus in modo
misurabile, al di là della sola emozione dominante.

### Nota metodologica
Il bias di aspettativa sul corpus *notizie* è in parte strutturale (il lessico)
e in parte tematico (il dominio). La pipeline adottata separa i due effetti:
le stopwords rimuovono il rumore lessicale, la normalizzazione riduce il bias
strutturale, quello che rimane è il segnale tematico reale del corpus.